# 02 · Ground-truth evaluation set
The archive only contains `Images/` (334 frames) and `masks/` (305 masks). The paper's **29 ground-truth frames** are the frames where the damage is not visible, so they have **no mask**. This notebook:

1. extracts those 29 frames (`Images` minus `masks`) and checks the counts,
2. finds the pot in each one with **Segment Anything** (box prompt),
3. creates the paper's four test masks per frame — *rectangle, marginal, large_marginal, multiple* — reproducibly (seeded),
4. writes preview sheets so you can check every mask by eye.

Use a **T4 GPU** runtime if you have one (faster); CPU also works. Resumable.

> Note: the 305 annotated masks cover only the *missing rim part*, not the whole pot, so they are not used to find the pot.

In [ ]:
# --- Bootstrap: mount Drive, get the code, make `heritage` importable ---
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/artefact-restoration-3d.git"  # <- set after you push to GitHub
REPO_DIR = Path("/content/artefact-restoration-3d")
DRIVE_CODE = Path("/content/drive/MyDrive/artefact-restoration-3d/code")   # fallback: upload the repo folder here

from google.colab import drive
drive.mount("/content/drive")

if "YOUR_USERNAME" not in REPO_URL:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif DRIVE_CODE.exists():
    # Re-sync from Drive on EVERY run so an updated code folder is never ignored
    shutil.copytree(DRIVE_CODE, REPO_DIR, dirs_exist_ok=True)
else:
    raise SystemExit("Set REPO_URL (after pushing to GitHub) or upload the repo folder to " + str(DRIVE_CODE))

sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
print("Repo ready at", REPO_DIR)


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import subprocess, sys
from heritage import paths
cfg = paths.load_config(); P = paths.get_paths(cfg).ensure()
# The dataset lives on the Colab VM disk (not Drive), so a new session needs it again.
# The downloader skips anything already present, so this is safe to run every time.
subprocess.run([sys.executable, 'scripts/download_dataset.py'], check=True)
assert (P.data / cfg['dataset']['images_subdir']).exists(), 'Dataset folder not found after download'

### Quick look at the annotated masks
Checks the mask file format (white-on-black or black-on-white) and how much of the frame the annotators covered.

In [ ]:
import cv2, numpy as np
from heritage.ids import list_images
from heritage.synth_masks import to_bool_mask
mdir = P.data / cfg['dataset']['masks_subdir']
files = list_images(mdir)
for f in files[:: len(files)//6][:6]:
    raw = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
    b = to_bool_mask(raw)
    print(f.name, raw.shape, raw.dtype, 'unique values:', np.unique(raw)[:4], f'-> covers {b.mean():.1%} of frame')

In [ ]:
!pip install -q segment-anything

In [ ]:
!python scripts/prepare_gt.py --method sam

### Check the result
Each row is one frame: **full frame with the pot outline (cyan)**, then zoomed crops with the mask in magenta: **rectangle · marginal · large_marginal · multiple**.
The cyan outline must follow the whole pot (not the stool, not the background), and every magenta region must lie on the pot.

In [ ]:
from PIL import Image
out = P.stage('gt')
for f in sorted((out / 'previews').glob('gt_masks_sheet_*.jpg')):
    print(f.name); display(Image.open(f))

In [ ]:
import json
man = json.loads((out / 'manifest.json').read_text())
print('GT frames :', man['n_gt'], man['gt_runs'])
print('damaged   :', man['n_damaged'], man['damaged_runs'])
rep = json.loads((out / 'mask_report.json').read_text())
bad = [r['frame'] for r in rep if not 0.05 < r['vase_frac_of_image'] < 0.8]
print('frames whose vase estimate looks suspicious:', bad or 'none')

### If something looks wrong
| Symptom | Fix |
|---|---|
| `ValueError: expected 29 GT frames` | Send me the printed message — do not edit the numbers in the config. |
| Cyan outline wrong for a few frames | Create `results/gt/vase_boxes.json` on Drive, e.g. `{"frame_00090": [0.30, 0.03, 0.85, 0.68]}` (relative x0,y0,x1,y1 of a box that tightly contains the pot) and re-run the cell. Only the changed frames are redone. |
| Outline wrong for most frames | Send me the sheets. Do not edit anything. |
| No GPU / SAM download fails | `!python scripts/prepare_gt.py --method grabcut` (CPU fallback, less accurate). |
| Old masks from the first version | Rebuilt automatically (settings stamp changed). |

**Send me** the sheets in `results/gt/previews/` and `manifest.json`. Nothing is inpainted until you confirm they look right.